# Voice Conductor: Guided First Tour

This notebook is a slower, friendlier walk through `voice-conductor`. We will make one small sound, look at the object that came back, and then keep reusing that same idea for routing, fallback, caching, exporting, hooks, and optional live providers.

Generated demo files stay flat in `demo_files/`. No per-run folder, no scavenger hunt.

## 1. Setup

Start with the few pieces we need. The local demo provider gives us a playful, offline voice with word timing metadata, which makes it a nice first step before we touch real devices or cloud APIs.

In [1]:

from IPython.display import Audio, display
from pathlib import Path
import pandas as pd

from voice_conductor._showcase_helpers import *
from voice_conductor.config import _DEFAULT_PROVIDER_CHAIN, Settings


demo_root = Path("demo_files")
demo_root.mkdir(exist_ok=True)

# Leave these as None unless you want to target a specific real output device.
PREFERRED_SPEAKER_DEVICE = None
PREFERRED_VIRTUAL_MIC_DEVICE = None
CONFIG_PATH = demo_root / "demo-voice_conductor.config.jsonc"

def load_config() -> Settings:
    config = Settings.from_file(CONFIG_PATH)
    config.voice_conductor.provider_chain = _DEFAULT_PROVIDER_CHAIN + ["demo"]
    config.save_settings(CONFIG_PATH)

    if PREFERRED_SPEAKER_DEVICE:
        config.voice_conductor.route_config.routes['speakers'].device = PREFERRED_SPEAKER_DEVICE
    if PREFERRED_VIRTUAL_MIC_DEVICE:
        config.voice_conductor.route_config.routes['mic'].device = PREFERRED_VIRTUAL_MIC_DEVICE
    
    return config

config = load_config()
config.voice_conductor.provider_chain

['elevenlabs', 'kokoro', 'azure', 'windows', 'demo']

## 2. Making Sound

Before routing, let’s ask the local demo provider for one small clip. The important thing is the return value: `synthesize_voice(...)` gives us a `SynthesizedAudio` object we can preview, inspect, save, or route later.

In [2]:
from voice_conductor import DemoProvider, TTSManager

# Changing the speed of the DemoProvider
config.providers.demo.speed = 1.4

demo_tts = TTSManager(settings=config)

first_text = "Notebook voice check: tiny blips, big personality."

voices = demo_tts._get_provider("demo").list_voices()
for each in voices[::-1]:
    print(name := each.id)
    first_audio = demo_tts.synthesize_voice(first_text, provider="demo", voice=name)
    display(audio_preview(first_audio))

demo:robot


demo:pilot


demo:animalese


## 3. Peek Inside The Object

That little audio widget came from a plain Python object. Here is the shape we will keep reusing: provider, voice, timing, samples, and metadata all travel together.

In [3]:
audio_summary(first_audio)

,provider,voice,duration_seconds,sample_rate,channels,frames,word_timing_items,style
0,demo,demo:animalese,2.192,16000,1,35069,7,retro talking blips


## 4. Find A Real Speaker

Now we ask the machine what it can actually play to. 

If this cell fails makes sure to install the audio extra with `pip install -e .[audio]`, connect or enable an output device, then rerun.

In [4]:
from voice_conductor.audio import list_output_devices

try:
    real_output_devices = list_output_devices()
except Exception as exc:
    raise RuntimeError(
        "The showcase plays audio through real output devices and could not "
        "query them. Install the audio extra with `pip install -e .[audio]`, "
        "then rerun the notebook with an available speaker or virtual cable."
    ) from exc

if not real_output_devices:
    raise RuntimeError(
        "The showcase did not find any real output devices. Connect or enable "
        "an output device, then rerun the notebook."
    )

pd.DataFrame(summarize_output_devices(real_output_devices)).style.hide(axis="index")

id,name,hostapi,default,virtual_cable,channels,sample_rate
5,Microsoft Sound Mapper - Output,MME,False,False,2,44100.000000
6,Speakers (High Definition Audio,MME,True,False,2,44100.000000
7,VoiceMeeter Input (VB-Audio Voi,MME,False,True,8,44100.000000
8,VoiceMeeter Aux Input (VB-Audio,MME,False,True,8,44100.000000
14,Primary Sound Driver,Windows DirectSound,False,False,2,44100.000000
15,Speakers (High Definition Audio Device),Windows DirectSound,False,False,2,44100.000000
16,VoiceMeeter Input (VB-Audio VoiceMeeter VAIO),Windows DirectSound,False,True,8,44100.000000
17,VoiceMeeter Aux Input (VB-Audio VoiceMeeter AUX VAIO),Windows DirectSound,False,True,8,44100.000000
18,VoiceMeeter Input (VB-Audio VoiceMeeter VAIO),Windows WASAPI,False,True,2,48000.000000
19,VoiceMeeter Aux Input (VB-Audio VoiceMeeter AUX VAIO),Windows WASAPI,False,True,2,48000.000000


## 5. Route The Same Audio

Nothing new gets synthesized here. We take the exact object from the first sound and send it through the selected speaker route.

In [5]:
from voice_conductor import PlaybackHooks, TTSManager


demo_tts = TTSManager()

_word_timings = lambda event: display(render_word_timing_transcript(event.audio, label="Demo provider word timing"))
hooks = PlaybackHooks(on_audio_ready=_word_timings)

## First, we synthesize audio
# first_text = "Notebook voice check: tiny blips, big personality."
# first_audio = demo_tts.synthesize_voice(first_text, provider="demo")

# Then, we route (or play) the audio to the configured speakers.
result = demo_tts.route(first_audio, routes="speakers", hooks=hooks)

pd.DataFrame(
    [
        {
            "route": route,
            "device": result.devices[route].name,
            "hostapi": result.devices[route].hostapi_name,
            "provider": result.audio.provider,
            "voice": result.audio.voice,
        }
        for route in result.routes
    ]
)


,route,device,hostapi,provider,voice
0,speakers,Speakers (High Definition Audio,MME,demo,demo:animalese


## 5.1 The Speak Method

Now, all at once...

In [6]:
tts = TTSManager(settings=config)

windows_audio = tts.speak(
    "This is a test of the voice conductor pipeline.", 
    provider="windows", voice="david",
    routes=["speakers", "mic"]
).audio

## 6. Export The Audio

The same object can leave the notebook as WAV or raw PCM16 bytes. The files go directly into `demo_files/` so they are easy to find after the tour.

In [7]:
wav_path = windows_audio.copy_to(demo_root / "guided-demo.wav")
pcm_path = windows_audio.copy_to(demo_root / "guided-demo.pcm", format="pcm16")

pd.DataFrame(
    [
        {"file": wav_path.name, "path": str(wav_path), "bytes": wav_path.stat().st_size},
        {"file": pcm_path.name, "path": str(pcm_path), "bytes": pcm_path.stat().st_size},
    ]
)

,file,path,bytes
0,guided-demo.wav,demo_files\guided-demo.wav,144866
1,guided-demo.pcm,demo_files\guided-demo.pcm,144822


## 7. Add Playback Hooks

Hooks are small callbacks around playback. They are handy when an app wants to update UI, write logs, or needs to do something the moment audio is ready.

In [8]:
from voice_conductor import PlaybackCompleteEvent, PlaybackHooks, PlaybackReadyEvent

hook_events = []

def on_audio_ready(event: PlaybackReadyEvent) -> None:
    hook_events.append({"event": "ready", "routes": ",".join(event.routes), "provider": event.audio.provider})
    display(render_word_timing_transcript(event.audio, label="Hook saw this audio"))


def on_playback_complete(event: PlaybackCompleteEvent) -> None:
    hook_events.append({"event": "complete", "routes": ",".join(event.routes), "status": "error" if event.error else "ok"})


hooked_audio = demo_tts.synthesize_voice("Hooks could be pretty valuable!", provider="demo", voice="demo:pilot")
hooked_result = demo_tts.route(
    hooked_audio,
    routes="speakers",
    hooks=PlaybackHooks(on_audio_ready=on_audio_ready, on_playback_complete=on_playback_complete),
)

pd.DataFrame(hook_events)

,event,routes,provider,status
0,ready,speakers,demo,NaN
1,complete,speakers,NaN,ok


## 8. Optional: Register A Tiny Provider Name

When an app wants a provider available by name, it can register one. This tiny registration still delegates to `DemoProvider`, so the example stays local and audible.

In [9]:
from voice_conductor import DemoProvider, TTSManager, register_provider, register_provider_config, unregister_provider, unregister_provider_config
from dataclasses import dataclass, field


@dataclass(slots=True)
class GuidedBonusProviderSettings:
    """Settings for the guided custom provider example."""

    default_voice: str = field(
        default="animalese",
        metadata={
            "doc": "Demo provider-local voice id used when synthesis does not specify one."
        },
    )
    speed: float = field(
        default=1.0,
        metadata={"doc": "Demo synthesis speed multiplier; 1.0 means normal speed."},
    )
    emphasis: float = field(
        default=1.0,
        metadata={"doc": "Example custom tuning knob for provider-specific behavior."},
    )
    

# Keep one stable id for provider registration, config keys, and synthesis calls.
CUSTOM_PROVIDER_ID = "guided_bonus"


def build_guided_bonus_provider(active_settings):
    """Wrap DemoProvider so we can register and use it under a custom name."""
    return DemoProvider(active_settings, name=CUSTOM_PROVIDER_ID, default_voice="demo:pilot")


# Register provider runtime behavior and the typed config shape it expects.
register_provider(CUSTOM_PROVIDER_ID, build_guided_bonus_provider)
register_provider_config(CUSTOM_PROVIDER_ID, GuidedBonusProviderSettings)


# Attach typed settings so save_settings(...) persists this custom provider section.
config.providers.extra[CUSTOM_PROVIDER_ID] = GuidedBonusProviderSettings(
    default_voice="pilot",
    speed=1.2,
    emphasis=2.0,
)

# Make this provider the first fallback candidate and the explicit default.
if CUSTOM_PROVIDER_ID not in config.voice_conductor.provider_chain:
    config.voice_conductor.provider_chain = [CUSTOM_PROVIDER_ID, *config.voice_conductor.provider_chain]
config.voice_conductor.default_provider = CUSTOM_PROVIDER_ID

# Use the same manager path as built-ins, but pin provider for a deterministic demo.
try:
    custom_provider_tts = TTSManager(settings=config)
    
    custom_provider_audio = custom_provider_tts.synthesize_voice(
        "Registered providers use the same manager path.",
        provider=CUSTOM_PROVIDER_ID,
    )
    display(audio_preview(custom_provider_audio))
    custom_provider_view = audio_summary(custom_provider_audio)

    # Persist the provider registration settings back to the notebook config file.
    custom_provider_tts.settings.save_settings(CONFIG_PATH)
finally:
    pass
    # unregister_provider(CUSTOM_PROVIDER_ID)
    # unregister_provider_config(CUSTOM_PROVIDER_ID)

custom_provider_view


,provider,voice,duration_seconds,sample_rate,channels,frames,word_timing_items,style
0,guided_bonus,demo:pilot,2.229,16000,1,35669,7,retro talking blips


## 9. Optional: ElevenLabs Live Path

This cell stays off by default because it can use real API quota. If you turn it on, the routing code stays familiar: synthesize one object, then route it.

In [ ]:
# Keep live cloud usage off unless you intentionally want to spend API quota.
RUN_LIVE_DEMO = True

config = load_config()
live_tts = TTSManager(settings=config)

if RUN_LIVE_DEMO:
    print(f"Config: {CONFIG_PATH}")
    if 'elevenlabs' not in live_tts.list_providers():
        live_view = pd.DataFrame([{"skipped": True, "reason": f"Set your credentials for elevenlabs: {CONFIG_PATH}"}])
    else:
        live_audio = live_tts.synthesize_voice("ElevenLabs is now wired through voice conductor.", provider="elevenlabs")
        live_result = live_tts.route(live_audio, routes="speakers")
        live_view = pd.DataFrame(
            [
                {
                    "provider": live_audio.provider,
                    "voice": live_audio.voice,
                    "duration_seconds": round(live_audio.duration_seconds, 3),
                    "played_to": live_result.devices["speakers"].name,
                }
            ]
        )
else:
    live_view = pd.DataFrame([{"skipped": True, "reason": "RUN_LIVE_DEMO is False."}])

live_view


Config: demo_files\demo-voice_conductor.config.jsonc


,provider,voice,duration_seconds,played_to
0,elevenlabs,"Roger - Laid-Back, Casual, Resonant",2.926,Speakers (High Definition Audio


## 10. What This Notebook Wrote

One last index keeps the flat workspace honest. The files below live directly under `demo_files/`.

In [11]:
pd.DataFrame(
    [
        {"file": path.name, "path": str(path), "bytes": path.stat().st_size}
        for path in sorted(demo_root.glob("guided*"))
        if path.is_file()
    ]
)

,file,path,bytes
0,guided-demo.pcm,demo_files\guided-demo.pcm,144822
1,guided-demo.wav,demo_files\guided-demo.wav,144866
